In [1]:
%load_ext autoreload
%autoreload 2


ModuleNotFoundError: No module named 'imp'

# Import Libraries

In [2]:
import os

import pandas as pd
import numpy as np
import nltk
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder


nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

# Read Data

In [3]:
path = "../"
train = pd.read_csv(os.path.join(path, "train.csv"))
test = pd.read_csv(os.path.join(path, "test.csv"))
print("Number of rows and columns in the train data set:", train.shape)
print("Number of rows and columns in the test data set:", test.shape)
train.head()

FileNotFoundError: [Errno 2] No such file or directory: '../train.csv'

# Label encoding

In [ ]:
le = LabelEncoder()

train.rate = le.fit_transform(train.rate)
train.head()

,rate,text
0,3,Очень понравилось. Были в начале марта с соба...
1,4,В целом магазин устраивает.\nАссортимент позво...
2,4,"Очень хорошо что открылась 5 ка, теперь не над..."
3,2,Пятёрочка громко объявила о том как она заботи...
4,2,"Тесно, вечная сутолока, между рядами трудно ра..."


# Text Preprocessing

In [ ]:
stopwords = nltk.corpus.stopwords.words("russian")

# Init tf-idf
vect_word = TfidfVectorizer(
    max_features=100,
    lowercase=True,
    analyzer="word",
    stop_words=stopwords,
    ngram_range=(1, 3),
    dtype=np.float32
)

In [ ]:
# Train tf-idf
X_train = vect_word.fit_transform(train["text"])
# Map tf-idf on test
X_test = vect_word.transform(test["text"])

y_train = train["rate"]

# Init Model

In [ ]:
# Init logreg model
logreg = LogisticRegression(
    C=2,
    random_state=42
)

# Train Model

In [ ]:
# Train logreg
logreg.fit(X_train, y_train)

/home/pc/Desktop/nlp_huawei_new2_task/venv/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(C=2, random_state=42)

# Create Predicts

In [ ]:
# Predict probabilities
preds_proba = logreg.predict_proba(X_test)
# Get classes
preds = np.argmax(preds_proba, axis=1)
# pred_labels = le.inverse_transform(preds)

# Create submission

In [ ]:
sample_submission = pd.read_csv(os.path.join(path, "sample_submission.csv"))
sample_submission["rate"] = preds
sample_submission.rate = le.inverse_transform(sample_submission.rate)
sample_submission.head()

,rate
0,5
1,5
2,5
3,3
4,5


In [ ]:
sample_submission.to_csv("submission.csv", index=False)